# Session 1: Retrieval-Augmented Generation (Student Notebook)

Welcome to Day 3! In this session, you will build production-grade RAG systems for **McKinsey's consulting knowledge management**. Moving beyond simple keyword matching, you will learn embedding-based retrieval, vector stores, advanced chunking, query transformation, and evaluation -- the techniques that power McKinsey's internal knowledge bases, engagement playbooks, and industry insight repositories.

## Learning Objectives

By the end of this session, you will be able to:

1. **Create and compare** text embeddings using OpenAI's embedding models on consulting knowledge artifacts
2. **Build and query** a vector store (ChromaDB) to power a McKinsey knowledge base
3. **Apply advanced chunking** strategies to consulting documents (strategy reports, engagement briefs, whitepapers)
4. **Transform queries** using HyDE and multi-query techniques for comprehensive industry analysis
5. **Build an end-to-end RAG pipeline** with source citations over McKinsey engagement materials
6. **Evaluate RAG quality** with retrieval and generation metrics for consulting use cases

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

# ============================================================
# Imports and Setup
# ============================================================

import openai
import chromadb
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.messages import HumanMessage, SystemMessage
import numpy as np
import json
import os

print("All imports successful!")

---
## Demos (Follow Along)

### Demo 1: Embedding Models -- Creating and Comparing Text Embeddings

Embeddings convert text into numerical vectors in a high-dimensional space where **semantically similar texts are close together**. This is the foundation of all modern retrieval systems.

In a consulting context, embeddings allow us to find relevant strategy insights, industry analyses, and engagement playbooks even when the exact terminology differs -- e.g., a query about "post-merger synergies" should retrieve documents discussing "integration value capture."

In [ ]:
# Demo 1 - Embedding Models (McKinsey Consulting Knowledge)

client = openai.OpenAI()

# Step 1: Create embeddings for McKinsey consulting knowledge snippets
texts = [
    "Digital transformation in financial services requires a phased approach starting with core banking modernization.",
    "Post-merger integration success depends on Day-1 readiness and cultural alignment between merging entities.",
    "Omnichannel retail strategy must prioritize seamless customer experience across physical and digital touchpoints.",
    "Supply chain resilience requires diversification of sourcing and near-shoring of critical components.",
    "The weather today is sunny and warm."
]

response = client.embeddings.create(
    model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
    input=texts
)

embeddings = [item.embedding for item in response.data]
print(f"Number of embeddings: {len(embeddings)}")
print(f"Embedding dimensions: {len(embeddings[0])}")
print(f"First 5 values of embedding 0: {embeddings[0][:5]}")

# Step 2: Compute cosine similarity
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("\nSimilarity matrix:")
print(f"{'':>45}", end="")
for i in range(len(texts)):
    print(f"  [{i}]", end="")
print()
for i, t1 in enumerate(texts):
    print(f"[{i}] {t1[:40]:>42}", end="")
    for j, t2 in enumerate(texts):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        print(f" {sim:.2f}", end="")
    print()

# Step 3: Semantic search — consulting query
query = "What are best practices for post-merger integration?"
query_emb = client.embeddings.create(model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=[query]).data[0].embedding

print(f"\nQuery: {query}")
print("Ranked results:")
scored = [(cosine_similarity(query_emb, emb), text) for emb, text in zip(embeddings, texts)]
scored.sort(reverse=True)
for score, text in scored:
    print(f"  {score:.4f} | {text}")

### Demo 2: Vector Stores -- Indexing and Similarity Search with ChromaDB

A vector store indexes embeddings for fast similarity search. ChromaDB is a lightweight, in-memory vector store perfect for prototyping.

In this demo, we build a **McKinsey knowledge base** -- indexing consulting insights across strategy, operations, M&A, digital transformation, and organizational design so consultants can quickly surface relevant prior work.

In [ ]:
# Demo 2 - Vector Stores with ChromaDB (McKinsey Knowledge Base)

# Step 1: Create a ChromaDB client and collection
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(
    name="mckinsey_knowledge_base",
    metadata={"hnsw:space": "cosine"}
)

# Step 2: Add McKinsey consulting knowledge documents with metadata
documents = [
    "Digital transformation in financial services requires a phased approach: digitize core processes, then build new digital offerings, and finally reimagine the business model.",
    "Post-merger integration should follow a 100-day plan covering Day-1 readiness, synergy capture, cultural integration, and operating model design.",
    "Omnichannel retail strategy must unify inventory, pricing, and customer data across all channels to deliver a seamless experience.",
    "Private equity value creation levers include revenue growth acceleration, operational efficiency, strategic M&A, and balance sheet optimization.",
    "Organizational restructuring requires a clear target operating model, role clarity, and a change management program to drive adoption.",
    "ESG strategy should be embedded into core business operations, not treated as a standalone compliance exercise, to create long-term shareholder value.",
    "Healthcare cost transformation requires addressing clinical variation, supply chain optimization, and administrative simplification simultaneously.",
    "Cross-border M&A transactions require careful due diligence on regulatory, tax, cultural, and operational integration risks."
]

# Use OpenAI embeddings via the API
client = openai.OpenAI()
emb_response = client.embeddings.create(model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=documents)
doc_embeddings = [item.embedding for item in emb_response.data]

collection.add(
    documents=documents,
    embeddings=doc_embeddings,
    ids=[f"doc_{i}" for i in range(len(documents))],
    metadatas=[
        {"source": "digital_transformation_playbook", "practice": "Digital"},
        {"source": "ma_integration_guide", "practice": "M&A"},
        {"source": "retail_strategy_whitepaper", "practice": "Retail"},
        {"source": "pe_value_creation_framework", "practice": "Private Equity"},
        {"source": "org_design_handbook", "practice": "Organization"},
        {"source": "esg_strategy_report", "practice": "Sustainability"},
        {"source": "healthcare_cost_study", "practice": "Healthcare"},
        {"source": "cross_border_ma_guide", "practice": "M&A"},
    ]
)
print(f"Indexed {collection.count()} documents in McKinsey knowledge base")

# Step 3: Query the collection — a typical consultant research question
query = "How should a retailer approach omnichannel transformation?"
query_emb = client.embeddings.create(model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=[query]).data[0].embedding

results = collection.query(
    query_embeddings=[query_emb],
    n_results=3
)

print(f"\nQuery: {query}")
print("Top 3 results:")
for i, (doc, distance, meta) in enumerate(zip(results['documents'][0], results['distances'][0], results['metadatas'][0])):
    print(f"  {i+1}. [{meta['source']} | {meta['practice']}] (dist={distance:.4f}) {doc}")

### Demo 3: Advanced Chunking Strategies

How you split documents directly affects retrieval quality. Different strategies work better for different content types: recursive splitting for general text, section-aware splitting for structured consulting reports, and sentence-based for executive summaries.

McKinsey documents typically have a structured format: **Executive Summary, Key Findings, Detailed Analysis, Recommendations, and Appendices**. Choosing the right chunking strategy preserves the logical structure of these sections.

In [ ]:
# Demo 3 - Advanced Chunking Strategies (McKinsey Consulting Report)

sample_doc = """# Post-Merger Integration: A Best Practice Guide

Post-merger integration (PMI) is the critical phase that determines whether an M&A transaction delivers its intended value. McKinsey research shows that 70% of mergers fail to achieve projected synergies, most often due to integration execution failures.

## Executive Summary

Successful post-merger integration requires a structured 100-day plan, early synergy identification, cultural alignment, and disciplined execution. Our analysis of 200+ transactions reveals that Day-1 readiness is the single strongest predictor of integration success.

## Key Findings

Three factors distinguish successful integrations from unsuccessful ones:

1. Leadership alignment: Joint leadership teams must be appointed within the first two weeks, with clear decision rights and accountability.
2. Synergy capture: Revenue and cost synergies must be quantified with bottom-up rigor and tracked through a dedicated Integration Management Office (IMO).
3. Cultural integration: Cultural differences must be proactively addressed through joint team workshops, shared values articulation, and visible leadership behavior.

## Recommendations

We recommend a phased approach to integration. Phase 1 (Days 1-30) focuses on stabilization and quick wins. Phase 2 (Days 31-100) addresses operating model design and synergy capture. Phase 3 (Months 4-12) drives full integration and transformation.

## Appendix

Detailed case studies from recent financial services and healthcare M&A transactions are provided in the supplementary materials, including synergy tracking templates and cultural assessment frameworks."""

# Strategy 1: Fixed-size recursive splitting
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=int(os.getenv("CHUNK_SIZE", "200")), chunk_overlap=int(os.getenv("CHUNK_OVERLAP", "30")), separators=["\n\n", "\n", ". ", " "]
)
recursive_chunks = recursive_splitter.split_text(sample_doc)

print(f"Recursive splitting: {len(recursive_chunks)} chunks")
for i, chunk in enumerate(recursive_chunks[:3]):
    print(f"  Chunk {i+1} ({len(chunk)} chars): {chunk[:80]}...")

# Strategy 2: Section-aware splitting (respects consulting report structure)
section_splitter = RecursiveCharacterTextSplitter(
    chunk_size=int(os.getenv("CHUNK_SIZE", "300")), chunk_overlap=int(os.getenv("CHUNK_OVERLAP", "30")),
    separators=["\n## ", "\n# ", "\n\n", "\n", ". ", " "]
)
section_chunks = section_splitter.split_text(sample_doc)

print(f"\nSection-aware splitting: {len(section_chunks)} chunks")
for i, chunk in enumerate(section_chunks[:3]):
    print(f"  Chunk {i+1} ({len(chunk)} chars): {chunk[:80]}...")

# Strategy 3: Sentence-level splitting (good for executive summaries)
sentence_splitter = RecursiveCharacterTextSplitter(
    chunk_size=int(os.getenv("CHUNK_SIZE", "150")), chunk_overlap=int(os.getenv("CHUNK_OVERLAP", "0")),
    separators=[". ", "\n", " "]
)
sentence_chunks = sentence_splitter.split_text(sample_doc)

print(f"\nSentence-level splitting: {len(sentence_chunks)} chunks")
for i, chunk in enumerate(sentence_chunks[:3]):
    print(f"  Chunk {i+1} ({len(chunk)} chars): {chunk[:80]}...")

print(f"\nComparison: Recursive={len(recursive_chunks)}, Section-aware={len(section_chunks)}, Sentence={len(sentence_chunks)}")

### Demo 4: Query Transformation Techniques

Consultants often write queries that are too vague or too specific for direct retrieval against the knowledge base. Query transformation improves retrieval by rewriting, expanding, or hypothetically answering the query before searching.

For example, a consultant asking "How do we help a bank go digital?" needs the system to also search for "core banking modernization," "digital customer experience," and "fintech partnership strategy."

In [ ]:
# Demo 4 - Query Transformation (McKinsey Consulting Research)

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

# Technique 1: Multi-query expansion for comprehensive industry analysis
def multi_query_expand(question, n=3):
    """Generate multiple alternative queries for better retrieval coverage."""
    response = llm.invoke([
        SystemMessage(content=f"You are a McKinsey consultant researching a topic. Generate {n} alternative versions of this research question. Each should approach it from a different angle (e.g., industry perspective, functional perspective, geographic perspective). Return as a JSON list of strings."),
        HumanMessage(content=question)
    ])
    try:
        queries = json.loads(response.content)
    except:
        queries = [question]
    return [question] + queries

# Technique 2: HyDE (Hypothetical Document Embeddings)
def hyde_transform(question):
    """Generate a hypothetical McKinsey insight, then use it for retrieval."""
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey senior partner. Write a short paragraph that would be a perfect answer to this consulting question. Write it as if it's from a McKinsey whitepaper -- authoritative, structured, and insight-driven."),
        HumanMessage(content=question)
    ])
    return response.content

# Test both techniques — typical consulting research question
question = "What are the key risks in cross-border M&A transactions?"

print(f"Original question: {question}")
print("\n--- Multi-Query Expansion ---")
expanded = multi_query_expand(question)
for i, q in enumerate(expanded):
    print(f"  {i+1}. {q}")

print("\n--- HyDE Transform ---")
hypothetical = hyde_transform(question)
print(f"  Hypothetical McKinsey insight: {hypothetical[:200]}...")
print("\n  (This hypothetical answer would be embedded and used for retrieval")
print("   instead of the original question, often finding more relevant chunks.)")

### Demo 5: End-to-End RAG Pipeline with Source Citations

This demo brings everything together into a complete RAG pipeline that retrieves relevant McKinsey knowledge artifacts, generates a consultant-ready answer, and includes source citations -- just as a consultant would reference prior engagement materials and published research in their deliverables.

In [ ]:
# Demo 5 - End-to-End RAG Pipeline with Citations (McKinsey Knowledge Base)

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
embed_client = openai.OpenAI()

# Step 1: Build McKinsey knowledge base
kb_docs = [
    {"text": "Post-merger integration success depends on Day-1 readiness. Our research across 200+ deals shows that companies with a structured 100-day plan capture 15-25% more synergies.", "source": "ma_integration_playbook.pdf", "page": 1},
    {"text": "Digital transformation in banking requires three horizons: digitize existing processes (H1), launch digital-native products (H2), and reimagine the business model around platforms (H3).", "source": "digital_banking_whitepaper.pdf", "page": 4},
    {"text": "Supply chain resilience starts with visibility. Tier-2 and Tier-3 supplier mapping, combined with scenario-based stress testing, reduces disruption impact by 30-40%.", "source": "supply_chain_resilience_report.pdf", "page": 7},
    {"text": "ESG-driven value creation requires embedding sustainability metrics into strategic planning, capital allocation, and executive compensation to move beyond compliance.", "source": "esg_strategy_guide.pdf", "page": 2},
    {"text": "Organizational health is the strongest predictor of long-term performance. Companies in the top quartile of OHI scores deliver 3x total returns to shareholders.", "source": "ohi_benchmark_study.pdf", "page": 12},
    {"text": "Revenue synergies in mergers are realized through cross-selling, pricing optimization, and geographic expansion, but require dedicated commercial integration teams to execute.", "source": "ma_integration_playbook.pdf", "page": 15}
]

# Step 2: Index in ChromaDB
chroma = chromadb.Client()
coll = chroma.create_collection(name="engagement_playbooks", metadata={"hnsw:space": "cosine"})

texts = [d["text"] for d in kb_docs]
embs = [e.embedding for e in embed_client.embeddings.create(model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=texts).data]
coll.add(
    documents=texts, embeddings=embs,
    ids=[f"doc_{i}" for i in range(len(kb_docs))],
    metadatas=[{"source": d["source"], "page": d["page"]} for d in kb_docs]
)

# Step 3: RAG function
def rag_query(question, k=3):
    # Retrieve
    q_emb = embed_client.embeddings.create(model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=[question]).data[0].embedding
    results = coll.query(query_embeddings=[q_emb], n_results=k)
    
    # Build context with source tags
    context_parts = []
    for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
        context_parts.append(f"[Source: {meta['source']}, p.{meta['page']}] {doc}")
    context = "\n\n".join(context_parts)
    
    # Generate
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey knowledge assistant. Answer the consultant's question based on the retrieved knowledge base context. Cite sources in [brackets]. If the context does not fully address the question, say so."),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion: {question}")
    ])
    
    return {"answer": response.content, "sources": results['metadatas'][0], "context": context}

# Step 4: Test with consulting queries
for q in ["What are best practices for post-merger integration?", "How should a bank approach digital transformation?", "What drives organizational health?"]:
    result = rag_query(q)
    print(f"Q: {q}")
    print(f"A: {result['answer']}")
    print(f"Sources: {[s['source'] for s in result['sources']]}\n")

---
## Tasks (Your Turn!)

### Task 1: Build an Embedding-Based Consulting Knowledge Search Engine

Create a search engine for McKinsey's consulting knowledge base. It should take a corpus of consulting insights (strategy reports, engagement briefs, industry analyses), embed them, and return the most relevant documents for a consultant's research query -- ranked by cosine similarity.

In [ ]:
# Task 1 - Build an Embedding-Based Consulting Knowledge Search Engine

# TODO: Build a SearchEngine class that:
# 1. Takes a list of consulting knowledge documents (strings) in __init__
# 2. Embeds all documents using OpenAI embeddings
# 3. Has a search(query, k) method that returns top-k results ranked by similarity
#
# Hint: Use openai.OpenAI().embeddings.create() to embed texts
# Hint: Compute cosine similarity between query embedding and all doc embeddings
# Hint: Return results sorted by similarity score (highest first)

class SearchEngine:
    def __init__(self, documents):
        # YOUR CODE HERE
        pass

    def search(self, query, k=3):
        # YOUR CODE HERE
        pass


# Test with McKinsey consulting knowledge corpus
corpus = [
    "Digital transformation in financial services requires a phased approach starting with core banking modernization.",
    "Post-merger integration success depends on Day-1 readiness and a structured 100-day plan for synergy capture.",
    "Omnichannel retail strategy must unify inventory, pricing, and customer data across physical and digital channels.",
    "Private equity value creation levers include revenue growth, operational efficiency, and strategic bolt-on acquisitions.",
    "Supply chain resilience requires diversification of sourcing, near-shoring critical components, and end-to-end visibility.",
    "Organizational health, as measured by the OHI, is the strongest predictor of sustained long-term performance.",
    "ESG strategy must be embedded into core business operations and capital allocation to create shareholder value.",
    "Cross-border M&A transactions require careful due diligence on regulatory, tax, and cultural integration risks."
]

# engine = SearchEngine(corpus)
# for query in ["What are best practices for post-merger integration?", "How should a retailer approach omnichannel transformation?", "What are the key risks in cross-border M&A?"]:
#     print(f"\nQuery: {query}")
#     results = engine.search(query, k=3)
#     for score, doc in results:
#         print(f"  {score:.4f} | {doc}")

### Task 2: Implement a Multi-Strategy Chunking Pipeline

Build a chunking pipeline that applies different strategies based on document type (markdown, code, plain text) and evaluates chunk quality.

In [ ]:
# Task 2 - Implement a Multi-Strategy Chunking Pipeline

# TODO: Create a SmartChunker class that:
# 1. Detects document type (markdown, code, plain text)
# 2. Applies appropriate splitting strategy for each type
# 3. Returns chunks with metadata (type, size, overlap)
#
# Hint: Check for "#" headers (markdown), "def "/"class " (code), else plain text
# Hint: Markdown: split on headers; Code: split on function/class boundaries; Plain: recursive
# Hint: Return a list of dicts: {"text": ..., "type": ..., "tokens": ...}

class SmartChunker:
    def __init__(self, chunk_size=int(os.getenv("CHUNK_SIZE", "300")), chunk_overlap=int(os.getenv("CHUNK_OVERLAP", "50"))):
        # YOUR CODE HERE
        pass

    def chunk(self, text):
        # YOUR CODE HERE
        pass


# Test
# chunker = SmartChunker()
# chunks = chunker.chunk("# Introduction\n\nSome text...")
# for c in chunks: print(f"[{c['type']}] {c['text'][:60]}...")

### Task 3: Create a Query Expansion and Reranking System

Build a retrieval system that expands the query into multiple variants, retrieves candidates for each, deduplicates, and reranks using the LLM.

In [ ]:
# Task 3 - Create a Query Expansion and Reranking System

# TODO: Build an AdvancedRetriever class that:
# 1. Expands the query into 3 alternative formulations
# 2. Retrieves candidates for each formulation from a ChromaDB collection
# 3. Deduplicates the combined results
# 4. Reranks using the LLM to score relevance (0-10)
#
# Hint: Use multi_query_expand from Demo 4
# Hint: Use a set to track seen documents for deduplication
# Hint: For reranking, ask the LLM to rate relevance of each doc to the query

class AdvancedRetriever:
    def __init__(self, collection):
        # YOUR CODE HERE
        pass

    def retrieve(self, query, k=3):
        # YOUR CODE HERE
        pass


# Test
# retriever = AdvancedRetriever(collection)
# results = retriever.retrieve("How to improve RAG accuracy?")
# for doc, score in results: print(f"{score}/10 | {doc[:80]}")

### Task 4: Build a Production RAG Pipeline with Evaluation Metrics

Build a complete RAG system that includes automated evaluation — measuring retrieval relevance, answer faithfulness (does the answer match the sources?), and completeness.

In [ ]:
# Task 4 - Build a Production RAG Pipeline with Evaluation

# TODO: Build an EvaluatedRAG class that:
# 1. Has a query() method that returns answer + sources + metrics
# 2. Evaluates retrieval relevance (are the retrieved chunks relevant?)
# 3. Evaluates answer faithfulness (is the answer supported by sources?)
# 4. Evaluates answer completeness (does it fully address the question?)
#
# Hint: Use LLM-as-judge to score each metric 1-5
# Hint: Return metrics as a dict: {"relevance": ..., "faithfulness": ..., "completeness": ...}
# Hint: Average the metrics for an overall quality score

class EvaluatedRAG:
    def __init__(self, documents):
        # YOUR CODE HERE
        pass

    def query(self, question):
        # YOUR CODE HERE
        pass

    def evaluate(self, question, answer, context):
        # YOUR CODE HERE
        pass


# Test
# rag = EvaluatedRAG(documents)
# result = rag.query("What is RAG?")
# print(f"Answer: {result['answer']}")
# print(f"Metrics: {result['metrics']}")

### Task 5: Metadata-Filtered Retrieval for Practice Area Search

Build a `MetadataFilteredRetriever` that restricts retrieval to specific practice areas or industries before running similarity search. This enables engagement managers to search only within relevant practice knowledge.

**Requirements:**
1. `add_documents(documents)` stores docs with metadata (practice_area, industry, doc_type)
2. `retrieve(query, filters, k)` applies ChromaDB `where` filters before similarity search
3. `cross_practice_search(query, practices, k_per_practice)` searches across selected practices and merges

In [ ]:
# Task 5 - Metadata-Filtered Retrieval for Practice Area Search

# TODO: Build a MetadataFilteredRetriever class that:
# 1. Stores documents with metadata (practice_area, industry, doc_type) in ChromaDB
# 2. Retrieves with optional metadata filters using ChromaDB where clauses
# 3. Supports cross-practice search that queries multiple practices and merges
#
# Hint: Use ChromaDB's where parameter: collection.query(..., where={"practice_area": {"$eq": "M&A"}})
# Hint: For multiple filters, use: where={"$and": [{...}, {...}]}
# Hint: cross_practice_search loops over practices, retrieves for each, then sorts merged results

class MetadataFilteredRetriever:
    def __init__(self, collection_name="mckinsey_filtered_kb"):
        # YOUR CODE HERE
        pass

    def add_documents(self, documents):
        """Add documents with metadata: {text, practice_area, industry, doc_type}."""
        # YOUR CODE HERE
        pass

    def retrieve(self, query, filters=None, k=3):
        """Retrieve with optional metadata filters."""
        # YOUR CODE HERE
        pass

    def cross_practice_search(self, query, practices, k_per_practice=2):
        """Search across multiple practice areas and merge results."""
        # YOUR CODE HERE
        pass


# Test
# docs = [
#     {"text": "Post-merger integration requires cultural alignment.", "practice_area": "M&A", "industry": "Cross-industry", "doc_type": "Framework"},
#     {"text": "Digital transformation depends on executive sponsorship.", "practice_area": "Digital", "industry": "Technology", "doc_type": "Playbook"},
#     {"text": "Supply chain resilience requires diversified sourcing.", "practice_area": "Operations", "industry": "Manufacturing", "doc_type": "Report"},
# ]
# retriever = MetadataFilteredRetriever()
# retriever.add_documents(docs)
# results = retriever.retrieve("integration challenges", filters={"practice_area": "M&A"})
# print(results)


### Task 6: Conversational RAG with Follow-Up Question Handling

Build a `ConversationalRAG` that handles multi-turn conversations. Follow-up questions like "What are the risks?" should be understood in context of the previous exchange (e.g., PMI risks, not general risks).

**Requirements:**
1. `ask(question)` maintains conversation history and returns answer with sources
2. Rewrites follow-up questions using conversation context before retrieval
3. Returns turn number, original question, rewritten search query, and answer

In [ ]:
# Task 6 - Conversational RAG with Follow-Up Question Handling

# TODO: Build a ConversationalRAG class that:
# 1. Indexes documents in ChromaDB (same pattern as previous tasks)
# 2. Maintains conversation history as a list of {role, content} dicts
# 3. Rewrites follow-up questions to be self-contained before retrieval
# 4. Generates answers using both retrieved context and conversation history
#
# Hint: Use LLM to rewrite follow-ups: "Given this conversation, rewrite the question to be self-contained"
# Hint: Skip rewriting for the first question (no history yet)
# Hint: Include last 4-6 history entries in the generation prompt for continuity
# Hint: Track turn number as len(history) // 2 + 1

class ConversationalRAG:
    def __init__(self, documents):
        # YOUR CODE HERE
        pass

    def _rewrite_query(self, question):
        """Rewrite follow-up questions to be self-contained."""
        # YOUR CODE HERE
        pass

    def ask(self, question):
        """Ask a question with conversation context."""
        # YOUR CODE HERE
        pass


# Test
# docs = [
#     "Post-merger integration (PMI) is critical in the first 100 days. Key risks include cultural misalignment and IT conflicts.",
#     "PMI best practices include a dedicated integration office and clear governance.",
#     "Supply chain digitization improves forecast accuracy by 20-30%.",
# ]
# conv_rag = ConversationalRAG(docs)
# r1 = conv_rag.ask("What are best practices for post-merger integration?")
# r2 = conv_rag.ask("What are the key risks?")  # Should understand PMI context
# print(f"Turn {r2['turn']}: search query = {r2['search_query']}")


---
## Optional / Bonus Tasks

The following tasks are for participants who finish early or want deeper practice.

### Optional Task 7: Hybrid Search — Combining Keyword and Semantic Retrieval

Build a `HybridSearchEngine` that combines keyword-based matching (for acronyms like OHI, PMI) with semantic search (for conceptual queries) using Reciprocal Rank Fusion.

**Requirements:**
1. `keyword_search(query, k)` — term-frequency based keyword matching
2. `semantic_search(query, k)` — embedding-based similarity search
3. `hybrid_search(query, k)` — fuse both rankings with Reciprocal Rank Fusion

In [ ]:
# Optional Task 7 - Hybrid Search (Keyword + Semantic)

# TODO: Build a HybridSearchEngine that:
# 1. Builds a keyword index (term -> document frequency) for keyword_search
# 2. Builds a ChromaDB index for semantic_search
# 3. Combines results with Reciprocal Rank Fusion (RRF)
#
# Hint: For keyword index, use re.findall(r"\w+", doc.lower()) and collections.Counter
# Hint: RRF formula: score(doc) = sum(1 / (k + rank)) across methods, where k=60 is typical
# Hint: Test with acronym queries (e.g., "OHI score") where keyword matching helps

class HybridSearchEngine:
    def __init__(self, documents):
        # YOUR CODE HERE
        pass

    def keyword_search(self, query, k=5):
        # YOUR CODE HERE
        pass

    def semantic_search(self, query, k=5):
        # YOUR CODE HERE
        pass

    def hybrid_search(self, query, k=5, rrf_k=60):
        # YOUR CODE HERE
        pass


# Test
# docs = [
#     "The OHI (Organizational Health Index) measures nine dimensions of organizational effectiveness.",
#     "Organizational health is a leading indicator of long-term performance.",
#     "The OHI score benchmarks companies against industry peers on leadership and culture.",
# ]
# engine = HybridSearchEngine(docs)
# print(engine.hybrid_search("OHI score benchmarks", k=3))


### Optional Task 8: RAG with Hallucination Detection and Source Grounding

Build a `GroundedRAG` that generates answers then verifies every claim is supported by retrieved sources. Ungrounded claims are flagged for partner review before delivery.

**Requirements:**
1. `verify_grounding(answer, sources)` checks each claim against sources using LLM-as-judge
2. `query(question)` returns answer + grounding report + flagged claims
3. Reports grounded ratio (e.g., 4/5 claims grounded = 80%)

In [ ]:
# Optional Task 8 - RAG with Hallucination Detection

# TODO: Build a GroundedRAG class that:
# 1. Retrieves context and generates an answer (same as previous RAG tasks)
# 2. Verifies each claim in the answer against the source documents
# 3. Returns a grounding report with grounded_ratio and flagged_claims
#
# Hint: Ask the LLM to structure its answer as numbered claims/bullets for easier verification
# Hint: Use LLM-as-judge to check each claim: "Is this claim supported by the sources?"
# Hint: Return JSON from the judge: {"claims": [{"claim": "...", "grounded": true/false}]}
# Hint: grounded_ratio = num_grounded / total_claims

class GroundedRAG:
    def __init__(self, documents):
        # YOUR CODE HERE
        pass

    def verify_grounding(self, answer, sources):
        """Check each claim in the answer against source documents."""
        # YOUR CODE HERE
        pass

    def query(self, question):
        """Retrieve -> Generate -> Verify grounding."""
        # YOUR CODE HERE
        pass


# Test
# docs = [
#     "McKinsey research shows that 70% of digital transformations fail due to lack of change management.",
#     "Companies with top-quartile OHI deliver 3x total returns to shareholders over 10 years.",
# ]
# grounded = GroundedRAG(docs)
# result = grounded.query("What are key success factors for digital transformation?")
# print(f"Grounding: {result['grounded_ratio']:.0%}")
# print(f"Flagged: {result['flagged_claims']}")


---
## Summary

In this session you learned production-grade RAG engineering:

1. **Embeddings** -- How text is converted to vectors where semantic similarity equals spatial proximity.
2. **Vector Stores** -- How ChromaDB indexes embeddings for sub-millisecond similarity search.
3. **Chunking Strategies** -- How splitting strategy directly affects retrieval quality.
4. **Query Transformation** -- How multi-query and HyDE improve retrieval for complex questions.
5. **End-to-End RAG** -- How to combine retrieval, generation, and source citations into a production pipeline.

**Up Next:** In Session 2, we will learn how to deploy and scale these systems in production.